# Circuit API experiments and custom graph passes

## Prerequisites

* Python 3.10+ in a virtual environment with **this branch's built `qdk` package**, `ipykernel`, and IPython installed. These circuit and pass APIs are experimental; a released package may not contain them.
* JupyterLab, or VS Code with its Python and Jupyter extensions. Select the Python kernel containing that QDK build.
* Permission to launch a local Python subprocess and write temporary files. The worker uses the kernel's own Python executable and installed QDK, not another environment.
* Optional GPU execution requires a QDK-supported GPU and drivers. It is disabled by default; no cloud account or native-language compiler is needed to run this notebook after installing QDK.

Complete the [introductory Python notebook](circuit_api_python.ipynb) first, or use the [setup guide](README.md). This notebook is self-contained: it implements the introduction's next experiments and adds two custom graph passes. All circuit and graph files are saved in binary and S-expression formats, their full paths are printed, and they remain available until the final cleanup cell.

Simulation still uses `to_qir` followed by `run_qir`. Running a pass or pipeline transforms a program; it does not simulate one.

In [ ]:
from collections import Counter
from pathlib import Path
from tempfile import TemporaryDirectory
import json
import math
import os
import subprocess
import sys

from IPython.display import Markdown, display
from qdk import Result, TargetProfile
from qdk.circuit import (
    PassAnchor,
    PassMatch,
    PassPatch,
    PassPipeline,
    QuantumCircuit,
    QuantumGraph,
)
from qdk.simulation import run_qir

SHOTS = 64
RUN_GPU = False  # Opt in; GPU errors never fall back to CPU.
work = TemporaryDirectory(prefix="qdk-python-experiments-")
directory = Path(work.name).resolve()
print("Artifact directory:", directory)


def counts_for(samples: list, width: int) -> Counter:
    assert len(samples) == SHOTS
    assert all(len(shot) == width for shot in samples)
    assert all(bit in (Result.Zero, Result.One) for shot in samples for bit in shot)
    return Counter(
        "".join("1" if bit == Result.One else "0" for bit in shot) for shot in samples
    )


def save_circuit(circuit: QuantumCircuit, name: str) -> None:
    binary = directory / f"{name}.qdk"
    text = directory / f"{name}.sexpr"
    binary.write_bytes(circuit.to_bytes())
    text.write_text(circuit.to_text(), encoding="utf-8")
    assert (
        QuantumCircuit.from_bytes(binary.read_bytes()).to_bytes() == circuit.to_bytes()
    )
    assert (
        QuantumCircuit.from_text(text.read_text(encoding="utf-8")).to_bytes()
        == circuit.to_bytes()
    )
    print(binary)
    print(text)

## A longer entangled register

Extend the Bell construction to a five-qubit Greenberger-Horne-Zeilinger (GHZ) state: H on the first qubit, then a chain of CX gates. Terminal Z measurements produce only `00000` or `11111`. The histogram checks correlations, not a proof of entanglement by itself; the state follows from the preparation circuit.

The full-state simulator's memory cost grows exponentially with qubit count, so keep this example small. Counts are statistical, not necessarily exactly half of each outcome.

In [ ]:
WIDTH = 5
ghz = QuantumCircuit(WIDTH, WIDTH)
ghz.h(0)
for target in range(1, WIDTH):
    ghz.cx(target - 1, target)
for qubit in range(WIDTH):
    ghz.measure(qubit, qubit)
print(ghz)

ghz_qir = ghz.to_qir(TargetProfile.Base)
ghz_counts = counts_for(run_qir(ghz_qir, shots=SHOTS, seed=42, type="cpu"), WIDTH)
assert set(ghz_counts) <= {"0" * WIDTH, "1" * WIDTH}
print("CPU GHZ counts:", ghz_counts)
save_circuit(ghz, "ghz")

## Adaptive feedback with an else body

Use three qubits and classical bits. The first measurement chooses a branch: One flips qubit 1; Zero flips qubit 2. In classical-bit order the possible outputs are `110` and `001`, respectively. The `otherwise()` context belongs to the same builder as the if-body; it is not a second independent condition.

The helper adds this feedback to an already prepared circuit. We use H preparation for random selection, then separately force each branch to check both bodies without depending on random coverage.

In [ ]:
def add_feedback(circuit: QuantumCircuit) -> QuantumCircuit:
    circuit.measure(0, 0)
    branch = circuit.if_test(0, 1)
    with branch:
        circuit.x(1)
    with branch.otherwise():
        circuit.x(2)
    circuit.measure(1, 1).measure(2, 2)
    return circuit


feedback = add_feedback(QuantumCircuit(3, 3).h(0))
print(feedback)
try:
    feedback.to_qir(TargetProfile.Base)
except ValueError as error:
    assert "Adaptive" in str(error)
    print("Expected Base refusal:", error)
else:
    raise AssertionError("Base must reject runtime feedback")

feedback_qir = feedback.to_qir(TargetProfile.Adaptive)
feedback_counts = counts_for(run_qir(feedback_qir, shots=SHOTS, seed=42, type="cpu"), 3)
assert set(feedback_counts) <= {"001", "110"}
print("CPU if/else counts:", feedback_counts)
save_circuit(feedback, "feedback_else")

In [ ]:
forced_zero = add_feedback(QuantumCircuit(3, 3))
zero_qir = forced_zero.to_qir(TargetProfile.Adaptive)
zero_counts = counts_for(run_qir(zero_qir, shots=SHOTS, type="cpu"), 3)
assert zero_counts == {"001": SHOTS}
print("Else body, first result forced to Zero:", zero_counts)

In [ ]:
forced_one = add_feedback(QuantumCircuit(3, 3).x(0))
one_qir = forced_one.to_qir(TargetProfile.Adaptive)
one_counts = counts_for(run_qir(one_qir, shots=SHOTS, type="cpu"), 3)
assert one_counts == {"110": SHOTS}
print("If body, first result forced to One:", one_counts)

## Save a symbolic template for another Python process

Create an unbound Rx template and save both snapshot formats. A separate process will load them, recover its own `theta` parameter, bind it to pi, compile, and simulate. Parameter handles belong to their circuit and process; only the snapshot bytes cross this boundary.

The worker writes bound snapshots and textual QIR back to disk. Its JSON report contains only the process ID, angle, and measurement counts; circuit persistence remains binary or S-expression, not JSON. Rx(pi) should always measure One, giving a deterministic handoff check.

In [ ]:
template = QuantumCircuit(1, 1)
theta = template.parameter("theta")
template.rx(theta, 0).measure(0, 0)
save_circuit(template, "symbolic")
assert template.parameters == [theta]

### The standalone worker

The next cell writes and displays the actual Python program that will run. It imports the installed QDK, not notebook variables or local helper modules. The code display uses fenced Markdown so the notebook renderer controls its theme.

In [ ]:
worker_source = """from collections import Counter
from pathlib import Path
import json
import os
import sys

from qdk import Result, TargetProfile
from qdk.circuit import QuantumCircuit
from qdk.simulation import run_qir

source = Path(sys.argv[1])
angle = float(sys.argv[2])
shots = int(sys.argv[3])
output = Path(sys.argv[4])

payload = source.read_bytes()
template = QuantumCircuit.from_bytes(payload)
text = source.with_suffix(".sexpr").read_text(encoding="utf-8")
assert QuantumCircuit.from_text(text).to_bytes() == payload
(theta,) = template.parameters
bound = template.assign_parameters([(theta, angle)])
assert bound.parameters == []
assert template.parameters == [theta]

(output / "bound.qdk").write_bytes(bound.to_bytes())
(output / "bound.sexpr").write_text(bound.to_text(), encoding="utf-8")
qir = bound.to_qir(TargetProfile.Base)
(output / "bound.ll").write_text(qir, encoding="utf-8")
samples = run_qir(qir, shots=shots, seed=42, type="cpu")
assert len(samples) == shots
assert all(len(shot) == 1 and shot[0] in (Result.Zero, Result.One) for shot in samples)
counts = Counter("1" if shot[0] == Result.One else "0" for shot in samples)
print(json.dumps({"pid": os.getpid(), "angle": angle, "counts": counts}))
"""
worker_path = directory / "bind_and_run.py"
worker_path.write_text(worker_source, encoding="utf-8")
print("Worker program:", worker_path)
display(Markdown(f"```python\n{worker_source}\n```"))

### Launch, load, and verify

Use `sys.executable` so the worker uses the same installed QDK as the notebook. Arguments are passed as a list, without a shell. Captured warnings and errors remain visible, and a failed exit stops the notebook rather than looking like a successful handoff.

In [ ]:
worker_directory = directory / "worker-output"
worker_directory.mkdir(exist_ok=True)
completed = subprocess.run(
    [
        sys.executable,
        str(worker_path),
        str(directory / "symbolic.qdk"),
        str(math.pi),
        str(SHOTS),
        str(worker_directory),
    ],
    cwd=directory,
    capture_output=True,
    text=True,
    check=False,
    timeout=60,
)
print(completed.stdout, end="")
print(completed.stderr, end="", file=sys.stderr)
completed.check_returncode()
report = json.loads(completed.stdout)
assert report["pid"] != os.getpid()
assert report["angle"] == math.pi
assert report["counts"] == {"1": SHOTS}
assert template.parameters == [theta]

bound_payload = (worker_directory / "bound.qdk").read_bytes()
worker_bound = QuantumCircuit.from_bytes(bound_payload)
worker_text = (worker_directory / "bound.sexpr").read_text(encoding="utf-8")
assert QuantumCircuit.from_text(worker_text).to_bytes() == bound_payload
assert worker_bound.parameters == []
worker_qir = (worker_directory / "bound.ll").read_text(encoding="utf-8")
assert worker_qir == worker_bound.to_qir(TargetProfile.Base)
for path in sorted(worker_directory.iterdir()):
    print(path)

## Custom passes on the graph

We will implement two semantics-preserving transformations:

1. A patch-returning rewrite removes identity gates, splicing the qubit's dataflow past each node.
2. A fragment-returning expansion decomposes X into the exact sequence H, Z, H, since HZH = X.

Matching and traversal stay native. Python receives a `PassMatch` record (node ID, operation name, operand count), not a mutable graph. Returning `None` declines a match. Both operations return a new graph and leave the input unchanged.

Graph verification checks structural correctness, not arbitrary mathematical equivalence of a user-written pass. Here the gate identities establish equivalence; the simulation below is an additional regression check. Removing a non-identity or substituting an arbitrary gate can change the program even if the graph still verifies.

### A phase-sensitive test circuit

Prepare a Bell state, apply X to its second qubit, and undo the Bell preparation before measurement. The result is deterministically `01`. Unlike measuring X on an isolated zero-state qubit, this interference check distinguishes X from Y, which would produce `11`. Two explicit identity gates give the removal pass something to simplify.

In [ ]:
probe = QuantumCircuit(2, 2)
probe.h(0).cx(0, 1).i(0).x(1).i(1).cx(0, 1).h(0)
probe.measure(0, 0).measure(1, 1)
original_circuit_bytes = probe.to_bytes()
graph = probe.to_graph()
original_graph_bytes = graph.to_bytes()
assert graph.diagnostics() == []
print("Graph operation counts:", Counter(node.operation for node in graph.nodes))
original_qir = graph.to_qir(TargetProfile.Base)
original_counts = counts_for(run_qir(original_qir, shots=SHOTS, type="cpu"), 2)
assert original_counts == {"01": SHOTS}
print("CPU original:", original_counts)

### Rewrite: remove identity nodes

Declare the operation names that the callback handles. `PassPatch.remove()` cannot insert a chain; that needs the separate expansion API. The reported application count counts accepted edits, while crossings count calls into Python.

In [ ]:
def remove_identity(match: PassMatch) -> PassPatch:
    assert match.operation == "quantum.i" and match.operands == 1
    return PassPatch.remove()


rewritten = graph.rewrite(["quantum.i"], remove_identity)
assert rewritten.outcome == "committed"
assert rewritten.applications == rewritten.crossings == 2
assert rewritten.graph.diagnostics() == []
assert all(node.operation != "quantum.i" for node in rewritten.graph.nodes)
assert graph.to_bytes() == original_graph_bytes
print("Rewrite:", rewritten.outcome, "applications:", rewritten.applications)
rewritten_qir = rewritten.graph.to_qir(TargetProfile.Base)
assert counts_for(run_qir(rewritten_qir, shots=SHOTS, type="cpu"), 2) == {"01": SHOTS}

### A bounded rewrite reports incomplete work

Run the same pass on the original graph with a one-application budget. One identity is removed and one remains. `budget_exhausted` is not a completed optimization; inspect the outcome before deciding whether to continue. The partial graph still verifies, and the original remains unchanged.

In [ ]:
limited = graph.rewrite(["quantum.i"], remove_identity, budget=1)
assert limited.outcome == "budget_exhausted"
assert limited.applications == 1
assert sum(node.operation == "quantum.i" for node in limited.graph.nodes) == 1
assert limited.graph.diagnostics() == []
assert graph.to_bytes() == original_graph_bytes
print("Bounded rewrite:", limited.outcome, "applications:", limited.applications)

### Expand: replace X with H, Z, H

An expansion returns a sequence of operation names, applied to the matched qubit. This example introduces no new parameters or operands. Assert the resulting circuit structure as well as the observed behavior, so the example demonstrates an actual transformation rather than a no-op.

In [ ]:
def decompose_x(match: PassMatch) -> list[str]:
    assert match.operation == "quantum.x" and match.operands == 1
    return ["quantum.h", "quantum.z", "quantum.h"]


rewritten_bytes = rewritten.graph.to_bytes()
expanded = rewritten.graph.expand(["quantum.x"], decompose_x)
assert expanded.outcome == "committed"
assert expanded.applications == expanded.crossings == 1
assert expanded.graph.diagnostics() == []
assert rewritten.graph.to_bytes() == rewritten_bytes
assert all(node.operation != "quantum.x" for node in expanded.graph.nodes)
raised = QuantumCircuit.from_graph(expanded.graph)
assert [instruction.name for instruction in raised] == [
    "h",
    "cx",
    "h",
    "z",
    "h",
    "cx",
    "h",
    "measure_z",
    "measure_z",
]
expanded_qir = expanded.graph.to_qir(TargetProfile.Base)
expanded_counts = counts_for(run_qir(expanded_qir, shots=SHOTS, type="cpu"), 2)
assert expanded_counts == {"01": SHOTS}
print("CPU after both custom passes:", expanded_counts)
print(raised)
save_circuit(raised, "custom_pass_circuit")

### Register the same callbacks in a pipeline

`GraphRewritten` is a graph anchor, reached after lowering. Passes at the same anchor run in registration order. We disable built-ins here to isolate our two transformations; the default pipeline includes compiler-provided simplifications.

The result carries both a graph and a raised circuit. We check that the pipeline and the two direct graph calls generate identical QIR, then run that QIR. The pipeline does not mutate its input circuit.

In [ ]:
pipeline = PassPipeline(built_ins=False)
pipeline.add_rewrite_pass(
    PassAnchor.GraphRewritten, "remove-identity", ["quantum.i"], remove_identity
)
pipeline.add_expand_pass(
    PassAnchor.GraphRewritten, "x-to-hzh", ["quantum.x"], decompose_x
)
print("Registered:", [(entry.anchor.name, entry.name) for entry in pipeline.listing()])
pipeline_result = pipeline.run(probe)
assert [entry.name for entry in pipeline_result.executed] == [
    "remove-identity",
    "x-to-hzh",
]
assert pipeline_result.changed == ["remove-identity", "x-to-hzh"]
assert pipeline_result.diagnostics == []
assert pipeline_result.graph.diagnostics() == []
assert pipeline_result.crossings == 3
assert probe.to_bytes() == original_circuit_bytes
pipeline_qir = pipeline_result.graph.to_qir(TargetProfile.Base)
assert pipeline_qir == expanded_qir
assert pipeline_result.circuit.to_qir(TargetProfile.Base) == pipeline_qir
assert counts_for(run_qir(pipeline_qir, shots=SHOTS, type="cpu"), 2) == {"01": SHOTS}
print("Changed passes:", pipeline_result.changed)

### Persist the transformed graph itself

Circuit and graph snapshots are different document types. Use `QuantumGraph.from_bytes` or `from_text` for graph snapshots, not the circuit loaders. Saving both lets another process resume graph inspection or compilation without repeating the custom transformations.

In [ ]:
graph_binary = directory / "custom_pass_graph.qdk"
graph_text = directory / "custom_pass_graph.sexpr"
graph_binary.write_bytes(pipeline_result.graph.to_bytes())
graph_text.write_text(pipeline_result.graph.to_text(), encoding="utf-8")
restored_graph = QuantumGraph.from_bytes(graph_binary.read_bytes())
restored_text_graph = QuantumGraph.from_text(graph_text.read_text(encoding="utf-8"))
assert (
    restored_graph.to_bytes()
    == restored_text_graph.to_bytes()
    == pipeline_result.graph.to_bytes()
)
assert restored_graph.diagnostics() == restored_text_graph.diagnostics() == []
assert restored_graph.to_qir(TargetProfile.Base) == pipeline_qir
print(graph_binary)
print(graph_text)

## Opt in to GPU execution

Set `RUN_GPU = True` in the setup cell and rerun the notebook. These calls explicitly request the GPU: failures remain visible, with no CPU fallback. We validate allowed outcomes rather than comparing CPU/GPU shot sequences. Both forced feedback branches and the phase-sensitive transformed program are included. These small circuits demonstrate the interface, not a GPU performance advantage.

In [ ]:
if RUN_GPU:
    examples = [
        ("GHZ", ghz_qir, WIDTH, {"0" * WIDTH, "1" * WIDTH}),
        ("if/else", feedback_qir, 3, {"001", "110"}),
        ("forced else", zero_qir, 3, {"001"}),
        ("forced if", one_qir, 3, {"110"}),
        ("worker-bound template", worker_qir, 1, {"1"}),
        ("original graph", original_qir, 2, {"01"}),
        ("expanded graph", expanded_qir, 2, {"01"}),
        ("custom pipeline", pipeline_qir, 2, {"01"}),
    ]
    for name, qir, width, allowed in examples:
        samples = run_qir(qir, shots=SHOTS, seed=42, type="gpu")
        counts = counts_for(samples, width)
        assert set(counts) <= allowed
        print(f"GPU {name}:", counts)
else:
    print("GPU execution disabled; set RUN_GPU = True to enable it.")

## Inspect or retain the files

All locations below are absolute. The symbolic and bound circuit snapshots, transformed graph snapshots, worker program, and QIR remain on disk until cleanup. Copy anything you want to keep outside this directory before running the final cell.

For more pass contracts and limitations, see the [circuit guide](../../../source/qdk_package/CIRCUIT-USAGE.md#writing-a-pass) and [pass extension points](../../../source/qdk_package/PASS-EXTENSION-POINTS.md).

In [ ]:
print("Artifact directory:", directory)
for path in sorted(directory.rglob("*")):
    if path.is_file():
        print(path)

## Cleanup

Remove only this notebook's temporary files. The original notebook, source checkout, and installed packages are not changed.

In [ ]:
work.cleanup()
assert not directory.exists()